# Vectors and matrices

Every model in this path is vectors and matrices underneath. A data point is a vector. A dataset is a matrix. A layer of a neural network is a matrix. This notebook covers what you need to read all of that fluently.

Order: intuition, maths, cost, from scratch, library, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

## 1. Intuition

A vector is an arrow, or equally a list of numbers. A house with 3 bedrooms, 2 bathrooms and 150 square metres is the vector `[3, 2, 150]`. Two houses are similar when their arrows point in a similar direction and have a similar length.

A matrix is a machine that takes one vector and returns another. It can stretch, rotate, shear or squash space. The key fact: the columns of the matrix are where the basis vectors land.

## 2. Maths

Dot product: $a \cdot b = \sum_i a_i b_i = \lVert a \rVert \lVert b \rVert \cos\theta$. It is large when two vectors agree, zero when they are perpendicular.

Norms: $\lVert x \rVert_2 = \sqrt{\sum x_i^2}$ (length), $\lVert x \rVert_1 = \sum |x_i|$ (taxicab distance).

Matrix product: $(AB)_{ij} = \sum_k A_{ik} B_{kj}$. Shapes must match as $(m \times k)(k \times n) = (m \times n)$.

Determinant: the factor by which $A$ scales area (2D) or volume (3D). A negative sign means a flip.

Inverse: $A^{-1}$ undoes $A$. It exists only when $\det A \neq 0$, which is the same as full rank.

Rank: the number of independent columns, which is the dimension of the space the matrix can reach.

## 3. Cost: why ML cares

The cost of a linear model is one matrix-vector product. With a data matrix $X$ of shape $(n, d)$ and weights $w$ of shape $(d,)$, the predictions are $\hat y = Xw$. That single line is the forward pass of linear regression. A dense neural network layer is the same line with a nonlinearity after it.

Cost in operations: $n \cdot d$ multiplications. 10,000 rows with 100 features is 1,000,000 multiply-adds, which NumPy does in well under a millisecond.

## 4. From scratch

Dot product and matrix multiplication with plain loops, checked against NumPy.

In [ ]:
def dot(a, b):
    total = 0.0
    for x, y in zip(a, b):
        total += x * y
    return total


def matmul(A, B):
    n, k = A.shape
    k2, m = B.shape
    assert k == k2, f"shape mismatch: {A.shape} @ {B.shape}"
    out = np.zeros((n, m))
    for i in range(n):
        for j in range(m):
            out[i, j] = dot(A[i, :], B[:, j])
    return out


a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, -5.0, 6.0])
A = rng.normal(size=(3, 4))
B = rng.normal(size=(4, 2))

print("dot:", dot(a, b), np.dot(a, b))
print("matmul matches:", np.allclose(matmul(A, B), A @ B))
cos = dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print("cosine similarity:", round(cos, 3))

### Matrix as a transformation

Take the unit square and the unit circle and push them through a matrix. The columns of $A$ show where $(1,0)$ and $(0,1)$ land. The determinant is the area scale.

In [ ]:
def transform_demo(A, title):
    theta = np.linspace(0, 2 * np.pi, 200)
    circle = np.vstack([np.cos(theta), np.sin(theta)])
    square = np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]])
    _, ax = plt.subplots(figsize=(4, 4))
    ax.plot(*circle, "C0", alpha=0.4)
    ax.plot(*square, "C0", alpha=0.4, label="before")
    ax.plot(*(A @ circle), "C1")
    ax.plot(*(A @ square), "C1", label="after")
    ax.set_aspect("equal")
    ax.grid(alpha=0.3)
    ax.legend()
    ax.set_title(f"{title}, det = {np.linalg.det(A):.2f}")
    plt.show()


transform_demo(np.array([[2.0, 0.0], [0.0, 1.0]]), "stretch x")
transform_demo(np.array([[1.0, 1.0], [0.0, 1.0]]), "shear")
transform_demo(np.array([[1.0, 2.0], [2.0, 4.0]]), "singular: collapses to a line")

The third matrix has determinant 0. Its second column is twice the first, so the whole plane lands on one line. Information is lost and no inverse can bring it back.

## 5. Library version

`numpy.linalg` gives norms, determinant, inverse, rank and a solver.

In [ ]:
M = np.array([[2.0, 1.0], [1.0, 3.0]])
print("det:", np.linalg.det(M))
print("inverse:\n", np.linalg.inv(M))
print("M @ inv(M):\n", M @ np.linalg.inv(M))
print("rank:", np.linalg.matrix_rank(M))

S = np.array([[1.0, 2.0], [2.0, 4.0]])
print("singular det:", np.linalg.det(S), "rank:", np.linalg.matrix_rank(S))

x = np.array([3.0, -4.0])
print("L2:", np.linalg.norm(x), "L1:", np.linalg.norm(x, 1))

Solving $Mx = b$: use `solve`, not `inv(M) @ b`. It is faster and more accurate.

In [ ]:
b = np.array([8.0, 13.0])
x = np.linalg.solve(M, b)
print("x:", x, "check:", M @ x)

### The data matrix and weights

In [ ]:
n, d = 5, 3
X = rng.normal(size=(n, d))
w = np.array([0.5, -1.0, 2.0])
y_hat = X @ w
print("X shape:", X.shape, "w shape:", w.shape, "y_hat shape:", y_hat.shape)
print("row 0 by hand:", dot(X[0], w), "vs", y_hat[0])

## 6. What breaks

Three things go wrong in practice.

1. Shape mismatch. `(3, 4) @ (3, 2)` fails. Read shapes before running.
2. Singular matrices. Duplicate or perfectly correlated features make $X^T X$ singular.
3. Near-singular matrices. The determinant is not zero, but the inverse is numerically huge and noisy.

In [ ]:
try:
    A @ A
except ValueError as err:
    print("shape error:", str(err)[:60])

try:
    np.linalg.inv(S)
except np.linalg.LinAlgError as err:
    print("singular:", err)

# near singular: tiny determinant, huge condition number
N = np.array([[1.0, 2.0], [1.0, 2.0 + 1e-10]])
print("det:", np.linalg.det(N), "condition number:", f"{np.linalg.cond(N):.2e}")

A condition number near $10^{10}$ means you lose about 10 of the roughly 16 digits a float64 holds. The next notebook uses this when fitting least squares.